In [1]:
# packages
#%pip install pandas

import pandas as pd
#%pip install scikit-learn

from mod02_build_bot_predictor import train_model

### Define a function to extract predictions from the model

In [2]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [3]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [4]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [5]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [6]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [7]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [8]:
confusion_matrix_and_metrics(y_train, y_pred_train)

{'tp': 363,
 'tn': 2637,
 'fp': 0,
 'fn': 0,
 'misclassification_rate': 0.0,
 'false_positive_rate': 0.0,
 'false_negative_rate': 0.0}

### Check results on the test set (new data not yet seen by the model)

In [9]:
confusion_matrix_and_metrics(y_test, y_pred_test)

{'tp': 30,
 'tn': 839,
 'fp': 35,
 'fn': 96,
 'misclassification_rate': 0.131,
 'false_positive_rate': 0.04004576659038902,
 'false_negative_rate': 0.7619047619047619}

# Discussion Questions

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

I just ran a few longer running builds, the longest of which took almost 35 minutes to train, many of which achieved perfect rates on the test data, but ultimately only achieved a misclassification rate of around .131 on the non-training data. I previously had misclassification rates of approx .122-.125 on the non-training data sets, which trained much faster and achieved a better rate. For the lnog build, there were almost as many false positives as true positives, which is an alarming outcome for a predictive model. For the quicker training models, most of the misclassifications were false negatives, which means that human users were usually unaffected by the misclassifications, but that about 10% of the bots evaded detection. As a first-level bot-banning filter, I would be reasonably confident in the quick models to do its job, as flagging anything over 75% of the active bots for only a few minutes of training seems like a generally efficient tradeoff. However, if the purpose of the test was for something similar to what is being done in our book (where bots that evade detection are a life-or-death threat to others), I would not be confident in the model to use it, as even radically increasing the robustess of the learning in a few longer training trials did not appear to have a major impact on reducing the misclassification rate, as the consequences of getting things wrong are far too serious to have misclassification rates over 10%. 

### What are potential ramifications of false positives from the model?

If the model is used to disable or deactivate accounts, the primary consequences would be human users having their access suspended, either temporarily or permanently, depending on the specific policies used and if there was an appeals process. Thus, the main ramification is loss of service for falsely flagged human users, with possible cascading effects of loss of consumer trust or loss of user base if the rate is high enough and the falsely flagged humans opt to not return or have no way to appeal a false positive. 

### What are potential ramifications of false negatives from the model?

The primary consequences of false negatives are that bot users are able to continue to perform their primary functions, which are typically malicious ones, whether that be misinformation spreading, flooding the zone, automated transactions, or various forms of data scraping in violation of most ToS and sometimes local laws as well. Broader consequences can include loss of consumer trust if the bots are able to significantly impact the quality of content, and on a more meta level, false negatives might also give bot designers insights on how to possibly evade the model more consistently, allowing them to create even more bots in a sort of evolution-like arms race.  